## Project Description

This is an Abstract Syntax Tree (AST) analysis project created by Khalid Mihlar.

### Project Goal

I am analyzing student submissions from Assignment 3 of the CS 2420 class. All student submissions are de-identified and anonymous. The primary goal is to examine how the `size` variable, located within the `ArrayCollection` function (which extends `Collection` in Java), is interacted with, updated, and mutated across different functions. I will be creating an AST and analyzing each node to identify these interactions.


In [2]:
from __future__ import annotations

import json
import re
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Set, Tuple

import javalang
import pandas as pd

# Set this to the folder that contains your .java files.
# "." means the same folder as the notebook's current working directory.
ROOT_DIR = Path("./inputs/submissions")
JAVA_GLOB = "*.java"

if not ROOT_DIR.exists():
    raise FileNotFoundError(f"Folder does not exist: {ROOT_DIR.resolve()}")

java_files = sorted(ROOT_DIR.glob(JAVA_GLOB))

print(f"Looking for Java files in: {ROOT_DIR.resolve()}")
print("Found files:")
for p in java_files:
    print(" -", p.name)

if len(java_files) == 0:
    print("\nNo .java files found.")
    print("Put your five selected Java files in the folder above,")
    print("or change ROOT_DIR to the correct folder path.")
else:
    print(f"\nConfirmed {len(java_files)} Java file(s) found.")

Looking for Java files in: \\wsl.localhost\Ubuntu\home\kmihlar\Projects\ast-project\inputs\submissions
Found files:
 - AfricanWildDog.java
 - Anthozoa.java
 - BlackRhinoceros.java
 - Newt.java
 - Yak.java

Confirmed 5 Java file(s) found.


# AST extraction rubric

A use of `size` is classified as follows.

### Read
Count as a read when `size` appears in:

- return expressions, comparisons, conditions, loop bounds
- array indices like `data[size]`
- right-hand side expressions
- method arguments
- unary or compound updates, because they consult the old value

### Write
Count as a write when `size` appears as the target of:

- assignment: `size = ...`
- unary update: `size++`, `++size`, `size--`, `--size`
- compound assignment: `size += 1`, `size -= 1`

### Read + write
Count as both when syntax implies both:

- `size++`
- `size--`
- `size += 1`
- `size = size + 1`
- `size = size - 1`

### Method-level aggregation
For each constructor and method, the notebook outputs:

- `reads_size`
- `writes_size`
- `write_kinds`
- `evidence`
- `local_shadowing_of_size`




In [3]:
@dataclass
class Evidence:
    kind: str #kind of evidence it is
    line: Optional[int] #source line number
    code: str #code snippet
    detail: str #extra information

@dataclass
class MethodSummary:
    method_id: str
    kind: str  # constructor or method
    reads_size: bool
    writes_size: bool
    read_kinds: List[str]
    write_kinds: List[str]
    local_shadowing_of_size: bool
    evidence: List[Dict[str, Any]]

@dataclass
class FileSummary:
    file: str
    class_name: Optional[str]
    declares_size_field: bool
    size_field_lines: List[int]
    methods: List[Dict[str, Any]]
    notes: List[str]


In [5]:
def load_source(path: Path) -> str:
    return path.read_text(encoding='utf-8')

def get_lines(src: str) -> List[str]:
    return src.splitlines()

def tree_setup(path: Path) -> FileSummary:
    src = load_source(path)
    lines = get_lines(src)
    notes: List[str] = []

    try:
        tree = javalang.parse.parse(src)
    except javalang.parser.JavaSyntaxError as e:
        return FileSummary(
            file=path.name,
            class_name=None,
            declares_size_field=False,
            size_field_lines=[],
            methods=[],
            notes=[f'Parse error: {e}'],
        )
    return tree
results = [tree_setup(path) for path in java_files]

